# Recommendation System

This notebook uses the best trained classifiers to identify client investment needs,
then maps those needs to product recommendations.

**Model outputs are loaded from pre-computed pickles** — no retraining needed.
Run all `utils/*.py` scripts first to populate `data/pickled_files/`.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

sys.path.insert(0, str(Path().resolve()))
from utils.preprocessing import FEATURE_NAMES, TARGETS, build_features, load_data, split_data

PICKLE_ROOT = Path('data/pickled_files')

## Best model outputs

Load the best model per target (update `BEST_MODELS` once `bestmodel_*.ipynb` identifies winners).

In [ ]:
# Update these keys to the winning model from bestmodel_*.ipynb
BEST_MODELS = {
    'IncomeInvestment':       'xgboost_shap',   # placeholder — check bestmodel_income.ipynb
    'AccumulationInvestment': 'hard_voting_ens', # placeholder — check bestmodel_accumulation.ipynb
}

best_results = {}
for target, model_key in BEST_MODELS.items():
    path = PICKLE_ROOT / model_key / f'{target.lower()}.joblib'
    if path.exists():
        best_results[target] = joblib.load(path)
        print(f'Loaded {model_key} for {target}')
    else:
        print(f'MISSING: {model_key}/{target.lower()}.joblib — run python -m utils.{model_key} first')

## Best model test-set metrics (reference)

In [ ]:
rows = []
for target, r in best_results.items():
    row = {'target': target, 'model': r.get('model_name', BEST_MODELS[target])}
    row.update({k: round(v, 3) for k, v in r['test_metrics'].items()})
    rows.append(row)

pd.DataFrame(rows).set_index('target')

## SHAP global importances (from XGBoost pickle)

In [ ]:
for target in TARGETS:
    xgb_path = PICKLE_ROOT / 'xgboost_shap' / f'{target.lower()}.joblib'
    if not xgb_path.exists():
        print(f'XGBoost pickle missing for {target}')
        continue

    r = joblib.load(xgb_path)
    shap_values = r['shap_values']
    shap_test_X = r['shap_test_X']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    plt.sca(axes[0])
    shap.summary_plot(shap_values, shap_test_X, plot_type='bar', show=False)
    axes[0].set_title(f'Mean |SHAP| — {target}')

    plt.sca(axes[1])
    shap.summary_plot(shap_values, shap_test_X, show=False)
    axes[1].set_title(f'SHAP Beeswarm — {target}')

    plt.tight_layout()
    plt.show()

---
## Recommendation system

> **To be implemented.** Use the loaded `best_results` models and SHAP outputs above
> as inputs to the recommendation logic below.

In [ ]:
# Load full dataset for inference
df = load_data()
X = build_features(df)
df.head()

In [ ]:
# --- Recommendation logic goes here ---